# 06 - Query-based detector (RT-DETR)

RT-DETR-l (Zhao et al., 2023, arXiv:2304.08069) from Ultralytics, COCO weights, trained with the
same protocol as the anchor-free runs in notebook 05: fit on folds 1-2 (every 10th frame,
ignored regions masked), 30 epochs, 1152 px, same seed and augmentation, evaluated by our COCO
evaluator on the original fold-0 and `val` frames.

**How bipartite matching removes NMS.** YOLO assigns several grid locations to each object
(task-aligned top-k), so after training several neighbouring predictions fire on the same
object and NMS has to delete the copies. RT-DETR predicts a fixed set of queries and the
Hungarian matcher assigns each ground-truth object to exactly one query (cost = 2 x focal class
cost + 5 x L1 + 2 x GIoU); every other query is trained as "no object". A query that duplicates
an already matched one is therefore penalised during training, and the decoder self-attention
lets queries see each other, so the network itself learns to output one box per object.

**How this changes training.** One positive per object (instead of about ten) gives a sparse
and, early in training, unstable signal: the matching of a query can switch between epochs.
RT-DETR adds denoising queries (noisy copies of ground-truth boxes that must be reconstructed,
DN-DETR) and auxiliary losses after every decoder layer to stabilise it. DETR-style models are
known to converge more slowly than dense CNN detectors.

| Run | Change against the previous step |
|---|---|
| qb0_rtdetr_l_1152 | RT-DETR-l, Ultralytics defaults (AdamW, lr 8.3e-4 for all parameters), 300 queries |
| qb1_bblr_1152 | paper recipe: lr 1e-4, backbone lr 0.1x, weight decay 1e-4 |
| qb2_q500_1152 | qb1 with 500 queries instead of 300 |
| qb3_bblr_nobiaswarm_1152 | qb1 without the bias warm-up (see below) |
| qb4_q500_default_1152 | qb0 with 500 queries instead of 300 |

**Bias warm-up.** During the warm-up Ultralytics raises every learning rate from 0 to lr0,
except the biases, which fall from `warmup_bias_lr` = 0.1 to lr0. With `optimizer=auto` (qb0
and all YOLO runs) it sets `warmup_bias_lr` to 0 because it picks AdamW; with `optimizer: AdamW`
written in the config (qb1, qb2) the value stays 0.1, and the biases are trained with AdamW at
lr 0.067 in epoch 1 and 0.033 in epoch 2 (column `lr/pg4` of `results.csv`), about 700 times the
intended 1e-4. qb1 and qb2 are kept as a failed attempt; qb3 and qb4 are the clean versions of
the two questions they were meant to answer.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.analysis import classify_errors, error_summary, paired_comparison, per_clip_ap
from src.common import CLASS_NAMES, REPO_ROOT, RUNS_DIR
from src.evaluate import coco_evaluate
from src.predict import EVAL_SETS
from src.train_tools import load_config

# Anchor-free runs trained with the same budget, used as the CNN reference
REFERENCE_RUNS = ["af1_imgsz1152", "af4_topk5_1152"]
pd.set_option("display.precision", 4)

## 1. Results on fold 0 and val

In [ ]:
def load_runs(family):
    found = {}
    for config_path in sorted((REPO_ROOT / "configs" / family).glob("*.yaml")):
        cfg = load_config(config_path)
        run_dir = RUNS_DIR / family / cfg["name"]
        if (run_dir / "metrics.json").exists():
            found[cfg["name"]] = {"cfg": cfg, "dir": run_dir,
                                  "metrics": json.loads((run_dir / "metrics.json").read_text())}
    return found


query_runs = load_runs("query_based")
anchor_free_runs = {k: v for k, v in load_runs("anchor_free").items() if k in REFERENCE_RUNS}
runs = {**anchor_free_runs, **query_runs}
print("query-based runs:", list(query_runs))

main_cols = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_500"]
rows = []
for name, run in runs.items():
    fold0, val = run["metrics"]["fold0"], run["metrics"]["val"]
    rows.append({"run": name, **{c: fold0[c] for c in main_cols},
                 "val_AP": val["AP"], "val_AP50": val["AP50"], "ms_per_image": val["ms_per_image"]})
results = pd.DataFrame(rows).set_index("run")
results

## 2. Validation mAP during training: RT-DETR against YOLO11

Ultralytics mAP@[.5:.95] after every epoch on every 20th fold-0 frame. Left: against epochs
(same number of passes over the data). Right: against training time on the same GPU, which is
the real compute budget.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for name, run in runs.items():
    curve = pd.read_csv(run["dir"] / "results.csv")
    curve.columns = curve.columns.str.strip()
    style = "--" if name in anchor_free_runs else "-"
    axes[0].plot(curve["epoch"], curve["metrics/mAP50-95(B)"], style, label=name)
    axes[1].plot(curve["time"] / 3600, curve["metrics/mAP50-95(B)"], style, label=name)
axes[0].set_xlabel("epoch")
axes[1].set_xlabel("training time, hours (RTX 4060 Laptop)")
axes[0].set_ylabel("mAP@[.5:.95] (Ultralytics, fold 0)")
axes[0].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.suptitle("Query-based (solid) vs anchor-free (dashed)")
plt.show()

## 3. Small objects and dense scenes

AP by COCO size, and AP on fold-0 frames grouped by the number of annotated objects in the
frame. Dense frames are where a fixed set of queries could become a limit.

In [ ]:
size_cols = ["AP_small", "AP_medium", "AP_large"]
ax = results[size_cols].plot.bar(figsize=(9, 4), width=0.8)
ax.set_ylabel("AP on fold 0")
ax.set_title("AP by COCO object size")
ax.tick_params(axis="x", rotation=20)
plt.show()

per_class = pd.DataFrame({name: {c: run["metrics"]["fold0"][f"AP_{c}"] for c in CLASS_NAMES}
                          for name, run in runs.items()}).T
per_class

In [ ]:
gt_path = EVAL_SETS["fold0"][1]
gt = json.loads(gt_path.read_text())
objects_per_image = pd.Series(0, index=[img["id"] for img in gt["images"]])
for ann in gt["annotations"]:
    if not ann["iscrowd"]:
        objects_per_image[ann["image_id"]] += 1

density_bins = [0, 20, 50, 100, np.inf]
density_names = ["<20", "20-50", "50-100", ">=100"]
density = pd.cut(objects_per_image, density_bins, labels=density_names, right=False)

density_rows = {}
for name, run in runs.items():
    detections = json.loads((run["dir"] / "predictions_fold0.json").read_text())
    density_rows[name] = {group: coco_evaluate(gt_path, detections, image_ids=ids.index)["AP"]
                          for group, ids in objects_per_image.groupby(density, observed=True)}
by_density = pd.DataFrame(density_rows).T[density_names]
print("frames per group:", density.value_counts().reindex(density_names).to_dict())
by_density

## 4. Error analysis

Same simplified TIDE as in notebook 05 (confidence >= 0.3). Without NMS, duplicates can only be
suppressed by the model itself, so the `duplicate` column directly shows how well one-to-one
matching was learned.

In [ ]:
error_rows = []
for name, run in runs.items():
    detections = json.loads((run["dir"] / "predictions_fold0.json").read_text())
    det_table, gt_table = classify_errors(gt_path, detections, score_thr=0.3)
    error_rows.append({"run": name, **error_summary(det_table, gt_table)})
errors = pd.DataFrame(error_rows).set_index("run")
errors["duplicate_share"] = errors["duplicate"] / (errors["tp"] + errors["duplicate"])
errors

## 5. Which differences are larger than the noise?

Per-clip AP on the 26 evaluation clips, paired against the first RT-DETR run and against the
YOLO11 run at the same resolution (see notebook 05, section 7).

In [ ]:
clip_ap = {}
for name, run in runs.items():
    clip_ap[name] = {}
    for eval_set in ("fold0", "val"):
        detections = json.loads((run["dir"] / f"predictions_{eval_set}.json").read_text())
        for clip, ap in per_clip_ap(EVAL_SETS[eval_set][1], detections).items():
            clip_ap[name][f"{eval_set}/{clip}"] = ap

for reference in ["qb0_rtdetr_l_1152", "af1_imgsz1152"]:
    if reference in clip_ap:
        comparison = paired_comparison(clip_ap, reference=reference).drop(index=reference)
        comparison["significant"] = comparison["mean_diff"].abs() >= 2 * comparison["std_err"]
        print("reference:", reference)
        display(comparison)

## 6. Findings

- **RT-DETR-l is clearly better than YOLO11s** at the same resolution and epochs: +0.054 mean
  per-clip AP over af1, better on 24 of 26 clips, better in every class and in every density
  group. The largest gain is recall, especially of small objects (0.59 vs 0.38 at score >= 0.3).
  The price is more false positives at a fixed threshold and twice the inference time.
- **300 queries are not a limit here**: AP is not lower in frames with 100 or more objects than
  in sparse frames (the densest frames are mostly parking lots with large cars).
- **Equal epochs or equal time give the same answer**: RT-DETR reaches its plateau after about
  2 epochs (0.2 h) and is above YOLO11s at every point of the time axis.
- **qb1 and qb2 failed because of the bias warm-up**, not because of their learning rates: the
  curve starts at 0, rises slowly until epoch 30, and 42% of the confident qb1 detections that hit
  an object are duplicates, i.e. the one-to-one matching has not been learned yet.
- **Without the warm-up the paper recipe is the best setting (qb3)**: val AP 0.262 against 0.253,
  +0.011 +- 0.004 per clip over qb0, better on 17 of 26 clips. Its curve rises until the last
  epoch instead of peaking after a few epochs and then falling like qb0, because the lower learning
  rate, and the even lower one for the backbone, change the pretrained features slowly.
- **500 queries do not help with the default settings (qb4)**: +0.003 +- 0.006 per clip, 12 clips
  better and 14 worse. Together with the density table, 300 queries are enough for VisDrone.